[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/translations/ru/notebooks/ch04_dynamic_programming_ru.ipynb)

<div style="background: linear-gradient(135deg, #0f1f0f 0%, #1a3a1a 50%, #2a5a2a 100%); padding: 40px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #7ee787; font-size: 2.2em; margin: 0 0 10px 0;">Глава 4. Динамическое программирование</h1>
  <p style="color: #a8dadc; font-size: 1.1em; margin: 0;">Оценивание стратегии, итерация стратегии и итерация ценности на сетке 4×4 — с анализом сходимости и тепловыми картами ценности.</p>
</div>

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #4fc3f7;">
  <strong style="color: #4fc3f7;">Подготовка окружения</strong><br>
  <span style="color: #cdd9e5;">Строим MDP GridWorld с нуля (внешняя RL-библиотека не нужна). Время выполнения менее 10 секунд.</span>
</div>

In [ ]:
!pip install numpy matplotlib --quiet

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import time

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
})
print('Setup complete.')

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #7ee787;">
  <strong style="color: #7ee787;">1. MDP «GridWorld 4×4»</strong>
</div>

Классический пример 4.1 из Sutton \& Barto: сетка 4×4, в которой два угловых состояния (верхний левый и нижний правый) терминальные. Все остальные переходы дают вознаграждение $-1$, что мотивирует агента как можно быстрее достичь терминального состояния.

Состояния 0–15 пронумерованы по строкам. Действия: Up(0), Down(1), Left(2), Right(3). Ходы за пределы сетки оставляют состояние без изменения.

In [ ]:
# ── Build GridWorld MDP ──

SIZE = 4
N_S = SIZE * SIZE    # 16 states
N_A = 4              # Up, Down, Left, Right
GAMMA = 1.0          # Undiscounted (episode ends at terminal)

TERMINALS = {0, 15}  # Top-left and bottom-right

def state_to_rc(s): return divmod(s, SIZE)
def rc_to_state(r, c): return r * SIZE + c

# Transition: P[s, a] = (next_state, reward)
MOVES = [(-1,0),(1,0),(0,-1),(0,1)]   # Up, Down, Left, Right

P = {}   # P[s][a] = list of (prob, next_state, reward, done)
for s in range(N_S):
    P[s] = {}
    for a in range(N_A):
        if s in TERMINALS:
            P[s][a] = [(1.0, s, 0.0, True)]
        else:
            dr, dc = MOVES[a]
            r, c = state_to_rc(s)
            nr = max(0, min(SIZE-1, r+dr))
            nc = max(0, min(SIZE-1, c+dc))
            ns = rc_to_state(nr, nc)
            P[s][a] = [(1.0, ns, -1.0, ns in TERMINALS)]

print('GridWorld MDP built.')
print(f'States: {N_S}, Actions: {N_A}, Terminals: {TERMINALS}')

In [ ]:
# ── Visualise the GridWorld layout ──
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(7, 7))
for s in range(N_S):
    r, c = state_to_rc(s)
    if s in TERMINALS:
        fc = '#2ea043'
        lbl = f'T\ns={s}'
    else:
        fc = '#21262d'
        lbl = str(s)
    rect = mpatches.FancyBboxPatch(
        (c + 0.05, (SIZE-1-r) + 0.05), 0.9, 0.9,
        boxstyle='round,pad=0.04', linewidth=1,
        edgecolor='#30363d', facecolor=fc
    )
    ax.add_patch(rect)
    ax.text(c + 0.5, (SIZE-1-r) + 0.5, lbl, ha='center', va='center',
            fontsize=11, color='white', fontweight='bold')
ax.set_xlim(0, SIZE); ax.set_ylim(0, SIZE)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title('4×4 GridWorld: State Layout (T = terminal)', fontsize=13)
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #7ee787;">
  <strong style="color: #7ee787;">2. Оценивание стратегии (итеративное)</strong>
</div>

При **равномерной случайной стратегии** $\pi(a|s) = 0{,}25$ для всех $a$ оцениваем $V^\pi$ итерациями:
$$V_{k+1}(s) \leftarrow \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a)\left[r + \gamma V_k(s')\right]$$

In [ ]:
# ── Policy Evaluation ──

def policy_evaluation(policy, P, n_s, n_a, gamma=1.0, theta=1e-4):
    """Evaluate a stochastic policy. policy[s] = array of action probs."""
    V = np.zeros(n_s)
    history = [V.copy()]
    iters = 0
    while True:
        delta = 0
        for s in range(n_s):
            v = 0
            for a in range(n_a):
                for prob, ns, r, done in P[s][a]:
                    v += policy[s][a] * prob * (r + gamma * V[ns])
            delta = max(delta, abs(v - V[s]))
            V[s] = v
        iters += 1
        history.append(V.copy())
        if delta < theta:
            break
    return V, history, iters


# Uniform random policy
random_policy = np.ones((N_S, N_A)) / N_A

t0 = time.time()
V_rand, hist_rand, iters_rand = policy_evaluation(random_policy, P, N_S, N_A, GAMMA)
t_rand = time.time() - t0

print(f'Policy evaluation converged in {iters_rand} iterations ({t_rand*1000:.1f} ms)')
print('V^π (uniform random policy):')
print(V_rand.reshape(SIZE, SIZE).round(1))

In [ ]:
# ── Visualise value function heatmap ──

def plot_value_heatmap(V, title, ax, policy=None, action_chars='↑↓←→'):
    grid = V.reshape(SIZE, SIZE)
    im = ax.imshow(grid, cmap='RdYlGn', aspect='equal')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for r in range(SIZE):
        for c in range(SIZE):
            s = rc_to_state(r, c)
            val_text = f'{grid[r,c]:.1f}'
            if policy is not None and s not in TERMINALS:
                best_a = np.argmax(policy[s])
                val_text += f'\n{action_chars[best_a]}'
            ax.text(c, r, val_text, ha='center', va='center',
                    fontsize=9, color='black', fontweight='bold')
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])


fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Show V at k=1, k=10, and convergence
plot_value_heatmap(hist_rand[1],  'After k=1 sweep',   axes[0])
plot_value_heatmap(hist_rand[10], 'After k=10 sweeps', axes[1])
plot_value_heatmap(V_rand,        f'Converged (k={iters_rand})', axes[2])

plt.suptitle('Policy Evaluation: Uniform Random Policy on 4×4 GridWorld', fontsize=13)
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #7ee787;">
  <strong style="color: #7ee787;">3. Итерация стратегии (Policy Iteration)</strong>
</div>

**Итерация стратегии** чередует:
1. **Оценивание стратегии**: вычисляет $V^{\pi_k}$ для текущей стратегии.
2. **Улучшение стратегии**: жадно — $\pi_{k+1}(s) = \arg\max_a Q^{\pi_k}(s,a)$.

Гарантированно сходится к $\pi^*$ за конечное число шагов.

In [ ]:
# ── Policy Iteration ──

def policy_improvement(V, P, n_s, n_a, gamma=1.0):
    """Return greedy policy and whether policy is stable."""
    new_policy = np.zeros((n_s, n_a))
    policy_stable = True
    for s in range(n_s):
        Q_s = np.zeros(n_a)
        for a in range(n_a):
            for prob, ns, r, _ in P[s][a]:
                Q_s[a] += prob * (r + gamma * V[ns])
        best_a = np.argmax(Q_s)
        new_policy[s, best_a] = 1.0
        if not np.argmax(random_policy[s]) == best_a:  # Simplified stable check
            policy_stable = False
    return new_policy, policy_stable


def policy_iteration(P, n_s, n_a, gamma=1.0):
    policy = np.ones((n_s, n_a)) / n_a   # Start uniform
    V_history = []
    pi_history = []
    iters = 0

    while True:
        V, _, _ = policy_evaluation(policy, P, n_s, n_a, gamma)
        V_history.append(V.copy())
        pi_history.append(policy.copy())
        new_policy, _ = policy_improvement(V, P, n_s, n_a, gamma)
        iters += 1
        if np.allclose(new_policy, policy):
            break
        policy = new_policy

    return V, policy, V_history, pi_history, iters


t0 = time.time()
V_pi, pi_opt_pi, V_hist_pi, pi_hist_pi, iters_pi = policy_iteration(P, N_S, N_A, GAMMA)
t_pi = time.time() - t0
print(f'Policy Iteration converged in {iters_pi} policy iterations ({t_pi*1000:.1f} ms)')
print('Optimal V* (Policy Iteration):')
print(V_pi.reshape(SIZE, SIZE).round(1))

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #7ee787;">
  <strong style="color: #7ee787;">4. Итерация ценности (Value Iteration)</strong>
</div>

**Итерация ценности** применяет оператор оптимальности Беллмана напрямую:
$$V_{k+1}(s) \leftarrow \max_a \sum_{s',r} p(s',r|s,a)\left[r + \gamma V_k(s')\right]$$

Объединяет оценивание и улучшение в один проход и часто сходится быстрее.

In [ ]:
# ── Value Iteration ──

def value_iteration(P, n_s, n_a, gamma=1.0, theta=1e-4):
    V = np.zeros(n_s)
    history = [V.copy()]
    delta_history = []

    while True:
        delta = 0
        for s in range(n_s):
            Q_s = np.zeros(n_a)
            for a in range(n_a):
                for prob, ns, r, _ in P[s][a]:
                    Q_s[a] += prob * (r + gamma * V[ns])
            v_new = Q_s.max()
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        history.append(V.copy())
        delta_history.append(delta)
        if delta < theta:
            break

    # Extract greedy policy
    policy = np.zeros((n_s, n_a))
    for s in range(n_s):
        Q_s = np.zeros(n_a)
        for a in range(n_a):
            for prob, ns, r, _ in P[s][a]:
                Q_s[a] += prob * (r + gamma * V[ns])
        policy[s, Q_s.argmax()] = 1.0

    return V, policy, history, delta_history


t0 = time.time()
V_vi, pi_opt_vi, V_hist_vi, delta_hist_vi = value_iteration(P, N_S, N_A, GAMMA)
t_vi = time.time() - t0
iters_vi = len(delta_hist_vi)
print(f'Value Iteration converged in {iters_vi} sweeps ({t_vi*1000:.1f} ms)')
print('Optimal V* (Value Iteration):')
print(V_vi.reshape(SIZE, SIZE).round(1))

In [ ]:
# ── Final comparison: value heatmaps + convergence ──
ACTION_CHARS = '↑↓←→'

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Optimal V* with policy arrows
plot_value_heatmap(V_vi, 'V* + Optimal Policy (VI)', axes[0], pi_opt_vi)
plot_value_heatmap(V_pi, 'V* + Optimal Policy (PI)', axes[1], pi_opt_pi)

# Convergence comparison
ax = axes[2]
ax.semilogy(delta_hist_vi, color='#ffd700', linewidth=2, label=f'Value Iteration ({iters_vi} sweeps)')

# For Policy Iteration, plot V change between PI iterations
if len(V_hist_pi) > 1:
    pi_deltas = [np.max(np.abs(V_hist_pi[i+1] - V_hist_pi[i]))
                 for i in range(len(V_hist_pi)-1)]
    ax.semilogy(pi_deltas, 'o-', color='#58a6ff', linewidth=2,
                label=f'Policy Iteration ({iters_pi} policy steps)')

ax.set_title('Convergence Comparison', fontsize=12)
ax.set_xlabel('Iteration / Policy Step')
ax.set_ylabel('max|ΔV| (log scale)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Dynamic Programming on 4×4 GridWorld', fontsize=14)
plt.tight_layout()
plt.show()

print(f'\nSummary:')
print(f'  Value Iteration:  {iters_vi:4d} sweeps,  {t_vi*1000:6.1f} ms')
print(f'  Policy Iteration: {iters_pi:4d} rounds,  {t_pi*1000:6.1f} ms')
print(f'  V* match: {np.allclose(V_vi, V_pi, atol=0.01)}')

<div style="background: #0d2137; padding: 20px; border-radius: 8px; border: 1px solid #1f6feb; margin-top: 20px;">
  <h3 style="color: #79c0ff; margin-top: 0;">Глава 4 — ключевые выводы</h3>
  <ul style="color: #c9d1d9; line-height: 1.8;">
    <li><strong>Оценивание стратегии</strong> сходится к $V^\pi$ повторным применением оператора Беллмана.</li>
    <li><strong>Итерация стратегии</strong> чередует оценивание и жадное улучшение — сходится за конечное число шагов.</li>
    <li><strong>Итерация ценности</strong> применяет оператор оптимальности Беллмана напрямую, объединяя оба шага — обычно меньше суммарных проходов.</li>
    <li>Оба метода требуют <strong>модели</strong> ($P$ и $R$) — это методы планирования с моделью.</li>
    <li>ДП — основа для безмодельных методов (МК, TD), которые снимают требование модели.</li>
  </ul>
</div>